# GR Output-Transformer LSTM — Amplitude-Matched Grey Box (Diff-SSL, no conditioning)

**Google Colab**: Runtime → **GPU**. Open via *File → Open notebook → GitHub*
(`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo for the
`06_conditioning` modules and mounts Drive for the dataset. **Push local changes before running.**

## Idea — split the compressor into a known envelope + a learned colorist

This models the compressor as a **grey box**:

```
dry ──(× exported GR gain envelope)──► amplitude-matched input ──► [LSTM] ──► wet
        amplitude matching (known)                                 output transformer (learned)
```

1. **Amplitude matching** reuses the *already-exported* GR curves
   (`gr_curves/<setting>/<song>.pt`, key `gr_db`): the matched input is
   `amp = dry · 10**(gr_db/20)`. This reproduces the compressor's **level and
   dynamics** exactly from a known signal — verified on real data to track the
   wet RMS envelope to **≈2 %** (corr(amp, wet)=0.997 vs corr(dry, wet)=0.989).
2. The **LSTM output transformer** then learns only the **residual nonlinear
   coloration** (harmonics, transient micro-shaping) the gain envelope can't
   produce — empirically just **~8 % RMS** of the signal. So it is an
   **audio→audio**, time-domain regressor with **no conditioning** (the GR curve
   already carries the setting).

### dB scaling & time alignment (handled in `amplitude_match.py`)
- `gr_db` is an **amplitude** ratio → invert with `10**(gr/20)` (not `/10`).
- `gr_db` and the audio share the **same causal sample grid** (both from the
  same 1024-sample trailing-RMS convention), so **no time shift** is applied.
- `gr_db` is clamped to `[-30, +5] dB` before exponentiating — guards the
  ill-conditioned silence tails, never touches real compression.

## What is identical to `05_conditioning/train_lstm_tfilm_gr.ipynb`
- **Dataset**: Diff-SSL-G-Comp, all **10 settings × 10 songs**, pooled as plain
  audio pairs (the "which setting" label is discarded — no conditioning).
- **Split**: `splits.py` (copied verbatim), **seed 42**, val = 1 song × all
  settings, test = held-out songs × lowest-threshold settings.

## What follows the SOTA waveform references (and differs from the GR head)
| | GR predictor (`03`/`05`) | This (output transformer) |
|---|---|---|
| **Target** | gain-reduction envelope (dB) | **wet output audio** |
| **Input** | dry | **amplitude-matched** dry |
| **Head** | 61-bin CREPE logits + soft-argmax | **single `Linear(H→1)`** + residual combiner |
| **Loss** | BCE / MSE on GR | **0.5·L1 + 0.5·MR-STFT** (+opt. ESR) — nablafx-diffssl / Optical-DRC |
| **Metrics** | L1 (dB) | **ESR, RMSE, MAE, MSE** |
| **Model** | frame-rate conv+LSTM | **sample-rate windowed LSTM** (Optical-DRC port, `02b`) |

Stateful TBPTT throughout: each (song, setting) track is streamed in
`segment_len` chunks with both LSTM states carried chunk→chunk (detached each
step), reset only at track boundary / epoch.

In [1]:
# -- 0. Dependencies ---------------------------------------------------
# No nablafx needed here (loss uses auraloss directly). Pin numpy first so the
# lightning install can't downgrade Colab's numpy 2.x and break torch.
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss lightning-utilities packaging
!pip install -q --no-deps lightning

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} - restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")

numpy 2.0.2, torch 2.11.0+cu128


In [2]:
# -- 1. Mount Drive (dataset) + clone repo from GitHub (code) ---------
# The repo is NOT synced to Drive (only data/ is). Code comes from GitHub -
# push local changes before (re)running this cell; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" pull --ff-only
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT

# Module directory for this notebook.
# NOTE: Colab clones from GitHub, so any *uncommitted* local folders won't exist there.
# The implementation for this notebook lives in `06_output/`.
CANDIDATE_DIRS = [
    os.path.join(REPO_ROOT, "06_output"),
    os.path.join(REPO_ROOT, "06_conditioning"),  # legacy / local-only fallback
]
COND_DIR = next((d for d in CANDIDATE_DIRS if os.path.isfile(os.path.join(d, "dataset.py"))), None)
assert COND_DIR is not None, (
    f"Could not find dataset.py in any of: {CANDIDATE_DIRS}. "
    f"Did the clone succeed? REPO_ROOT={REPO_ROOT}"
)

OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "output_transformer_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), f"Bad DATA_ROOT: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, "processed_ground_truth")), "Missing wet audio dir"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# repo root (for `src`) + module dir (for dataset/model/system/splits)
for p in (REPO_ROOT, COND_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"COND_DIR   : {COND_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
REPO_ROOT  : /content/Virtual-Analogue-Compressor-Modelling
COND_DIR   : /content/Virtual-Analogue-Compressor-Modelling/06_output
DATA_ROOT  : /content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp
OUTPUT_DIR : /content/drive/Othercomputers/MacBook Air/data/output_transformer_runs


In [3]:
# -- 2. Cache dataset to Colab local SSD ------------------------------
# The output transformer needs dry + GR curves (to build the matched input)
# AND the wet audio (the target). Mirror 05's cache and add the wet WAVs.

import shutil
from dataset import discover_output_transformer_pairs

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"

pairs = discover_output_transformer_pairs(DATA_ROOT)
settings = sorted({p["setting"] for p in pairs})
songs = sorted({p["song"] for p in pairs})
print(f"Caching {len(songs)} songs x {len(settings)} settings ({len(pairs)} pairs) -> {LOCAL_DATA_ROOT}")

def _mirror(src: Path, dst: Path):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

# dry WAVs (one per song, shared across settings)
for song in songs:
    fn = f"{song}_UnmasteredWAV.wav"
    _mirror(Path(DATA_ROOT) / "processed_normalized" / fn,
            Path(LOCAL_DATA_ROOT) / "processed_normalized" / fn)

# GR curves (.pt) + wet WAVs (-exported.wav), per (song, setting) pair
for p in pairs:
    _mirror(p["gr"], Path(LOCAL_DATA_ROOT) / "gr_curves" / p["setting"] / Path(p["gr"]).name)
    _mirror(p["wet"], Path(LOCAL_DATA_ROOT) / "processed_ground_truth" / p["setting"] / Path(p["wet"]).name)

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

Caching 10 songs x 10 settings (100 pairs) -> /content/Diff-SSL-G-Comp
Using local cache: /content/Diff-SSL-G-Comp


In [ ]:
# -- 3. Imports & hyper-parameters ------------------------------------

import json
from datetime import datetime

import torch
import lightning as pl
from lightning.pytorch.callbacks import (
    EarlyStopping, LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

from dataset import (
    SAMPLE_RATE, SEGMENT_LEN, WINDOW,
    OutputTransformerDataModule, discover_output_transformer_pairs,
)
from model import OutputTransformerLSTM
from system import OutputTransformerSystem
from splits import build_split_manifest
from amplitude_match import GR_DB_MIN, GR_DB_MAX

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split (identical to 05) --
SPLIT_SEED   = 42
N_VAL_SONGS  = 1
N_TEST_SONGS = 2

# -- training --
LR                  = 1e-3
MAX_EPOCHS          = 100       # committed budget == CosineAnnealingLR T_max
EARLY_STOP_PATIENCE = 200
SCHEDULER           = "cosine"  # or "plateau" (ReduceLROnPlateau, nablafx recipe)
ETA_MIN             = 1e-6
WARMUP_SAMPLES      = 0         # drop N samples on each track's fresh-state chunk

# -- model (matched to the 02b SOTA Optical-DRC LSTM, conditioning removed) --
# Conv1d(1,2,k=64) -> LSTM(2,6) -> Linear(6,2) -> LSTM(2,6) -> Linear(6,1).
# encoder_activation="none" reproduces the bare 02b front-end (no PReLU). The
# output head stays "residual_add": the amplitude-matched input already carries
# level+dynamics, so the net only learns the nonlinear coloration (filtering /
# saturation) on top of it.
ENCODER_CHANNELS   = 2          # 02b: Conv1d(1, 2, k=64)
HIDDEN_SIZE        = 6          # 02b: LSTM units
MID_CHANNELS       = 2          # 02b: Dense(2) between the two LSTMs
NUM_LSTM_LAYERS    = 2
ENCODER_ACTIVATION = "none"     # "prelu" | "tanh" to add a front-end nonlinearity
OUTPUT_MODE        = "residual_add"   # "residual_gain" | "direct" for ablation
OUT_ACTIVATION     = "none"           # "tanh" to bound the output/correction

# -- loss (SOTA waveform recipe) --
L1_WEIGHT     = 0.5
MRSTFT_WEIGHT = 0.5
ESR_WEIGHT    = 0.0   # set > 0 to add the Optical-DRC / comparative-study ESR term

RUN_TAG    = "lstm_output_transformer_ampmatched"
RESUME_RUN = None

In [5]:
# -- 4. Preview split (must match 05) ---------------------------------

preview = build_split_manifest(
    discover_output_transformer_pairs(DATA_ROOT),
    seed=SPLIT_SEED, n_val_songs=N_VAL_SONGS, n_test_songs=N_TEST_SONGS,
)
print(f"Settings ({len(preview.all_settings)}): {preview.all_settings}")
print(f"Test settings (lowest T): {preview.test_settings}")
print(f"Train songs: {preview.train_songs}")
print(f"Val songs  : {preview.val_songs}")
print(f"Test songs : {preview.test_songs}")
print(f"Pairs - train={len(preview.train_pair_keys)} "
      f"val={len(preview.val_pair_keys)} test={len(preview.test_pair_keys)}")

Settings (10): ['threshold_-12_attack_10_release_0.4_ratio_10', 'threshold_-12_attack_1_release_0.1_ratio_2', 'threshold_-4_attack_10_release_0.1_ratio_2', 'threshold_-4_attack_1_release_0.4_ratio_10', 'threshold_-8_attack_30_release_0.8_ratio_4', 'threshold_0_attack_3_release_0.8_ratio_4', 'threshold_12_attack_3_release_0.8_ratio_2', 'threshold_4_attack_10_release_0.1_ratio_10', 'threshold_8_attack_1_release_0.1_ratio_10', 'threshold_8_attack_30_release_0.4_ratio_2']
Test settings (lowest T): ['threshold_-12_attack_10_release_0.4_ratio_10', 'threshold_-12_attack_1_release_0.1_ratio_2']
Train songs: ['BackroomInTulsa', 'Borderline', 'Electrvm', 'LivingLie', 'NosPalpitants', 'OpenFire', 'SongForJohn']
Val songs  : ['Ecstasy']
Test songs : ['Air', 'AncoraQui']
Pairs - train=70 val=10 test=4


In [ ]:
# -- 5. Model size ----------------------------------------------------

model = OutputTransformerLSTM(
    window=WINDOW, encoder_channels=ENCODER_CHANNELS, hidden_size=HIDDEN_SIZE,
    mid_channels=MID_CHANNELS, num_lstm_layers=NUM_LSTM_LAYERS,
    encoder_activation=ENCODER_ACTIVATION,
    output_mode=OUTPUT_MODE, out_activation=OUT_ACTIVATION,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"OutputTransformerLSTM: {n_params:,} params  (output_mode={OUTPUT_MODE})")
for name, mod in model.named_children():
    print(f"  {name:12s} {sum(p.numel() for p in mod.parameters()):,}")
print(f"\nWindow {WINDOW} samples | segment {SEGMENT_LEN} ({SEGMENT_LEN/SAMPLE_RATE:.2f}s) | {SAMPLE_RATE} Hz")

In [ ]:
# -- 6. Train ---------------------------------------------------------

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"ot_lstm_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = OutputTransformerDataModule(
    data_root=DATA_ROOT, segment_len=SEGMENT_LEN, window=WINDOW,
    sample_rate=SAMPLE_RATE, split_seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS, n_test_songs=N_TEST_SONGS,
    split_manifest_path=split_path,
)
dm.setup()
print(f"Train/val/test streams: {dm.train_dataset.B} / {dm.val_dataset.B} / {dm.test_dataset.B}")
print(f"Steps/epoch (train): {len(dm.train_dataset)}")

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "gr_output_transformer_amplitude_matched",
        "model_type": "sample_rate_windowed_lstm",
        "source_model": "Optical-DRC create_model_LSTM (no conditioning), 02b-matched dims",
        "dataset": "Diff-SSL-G-Comp", "settings": "all 10 (pooled, no conditioning)",
        "input": "amplitude_matched = dry * 10**(clamp(gr_db, %g, %g)/20)" % (GR_DB_MIN, GR_DB_MAX),
        "target": "wet audio (direct)", "sample_rate": SAMPLE_RATE,
        "window": WINDOW, "segment_len": SEGMENT_LEN,
        "split_seed": SPLIT_SEED, "train_songs": dm.split.train_songs,
        "val_songs": dm.split.val_songs, "test_songs": dm.split.test_songs,
        "test_settings": dm.split.test_settings,
        "model": {"encoder_channels": ENCODER_CHANNELS, "hidden_size": HIDDEN_SIZE,
                   "mid_channels": MID_CHANNELS, "num_lstm_layers": NUM_LSTM_LAYERS,
                   "encoder_activation": ENCODER_ACTIVATION,
                   "output_mode": OUTPUT_MODE, "out_activation": OUT_ACTIVATION,
                   "num_params": n_params},
        "loss": {"l1": L1_WEIGHT, "mrstft": MRSTFT_WEIGHT, "esr": ESR_WEIGHT,
                  "ref": "nablafx-diffssl 0.5*L1+0.5*MR-STFT (+opt ESR)"},
        "metrics": ["esr", "rmse", "mae", "mse"],
        "lr": LR, "max_epochs": MAX_EPOCHS, "scheduler": SCHEDULER,
        "eta_min": ETA_MIN, "warmup_samples": WARMUP_SAMPLES,
        "early_stop_patience": EARLY_STOP_PATIENCE,
    }, f, indent=2)

system = OutputTransformerSystem(
    model=model, lr=LR, l1_weight=L1_WEIGHT, mrstft_weight=MRSTFT_WEIGHT,
    esr_weight=ESR_WEIGHT, warmup_samples=WARMUP_SAMPLES, scheduler=SCHEDULER,
    max_epochs=MAX_EPOCHS, eta_min=ETA_MIN,
)


class ResumeOverrides(pl.Callback):
    # On resume, checkpoint restore overwrites cosine T_max and EarlyStopping
    # state with the old run's values - re-apply the notebook hparams so an
    # extended budget and a fresh early-stop window actually take effect.

    def on_train_start(self, trainer, pl_module):
        sched = trainer.lr_scheduler_configs[0].scheduler
        if hasattr(sched, "T_max"):
            sched.T_max = MAX_EPOCHS
        for cb in trainer.callbacks:
            if isinstance(cb, EarlyStopping):
                cb.patience = EARLY_STOP_PATIENCE
                cb.wait_count = 0
                cb.best_score = torch.tensor(float("inf"))


ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(dirpath=ckpt_dir, monitor="loss/val", mode="min", save_top_k=3,
                    save_last=True, filename="best-{epoch:03d}-{step}",
                    auto_insert_metric_name=False),
    LearningRateMonitor(logging_interval="epoch"),
    EarlyStopping(monitor="loss/val", mode="min", patience=EARLY_STOP_PATIENCE, verbose=True),
    TQDMProgressBar(refresh_rate=10),
    ResumeOverrides(),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
    precision="32-true",            # tiny model; fp32 keeps stateful carry + MR-STFT stable
    callbacks=callbacks, logger=loggers,
    gradient_clip_val=1.0, gradient_clip_algorithm="norm",
    log_every_n_steps=10, use_distributed_sampler=False,
)
trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")

In [ ]:
# -- 7. Test (held-out songs x lowest-threshold settings) -------------

trainer.test(system, datamodule=dm, ckpt_path=callbacks[0].best_model_path)

In [ ]:
# -- 8. Plot: amplitude-matched baseline vs prediction vs target ------
# Streams the val set in order so the LSTM state settles, then plots a
# mid-track chunk per stream. The amplitude-matched input is the grey-box
# baseline; the gap from it to the target is what the transformer learns.

import matplotlib.pyplot as plt
import numpy as np
from system import esr_metric, _detach_state

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

val_steps = list(dm.val_dataloader())
pick = len(val_steps) // 2

state = None
with torch.no_grad():
    for s, (inp, wet, mask, reset) in enumerate(val_steps):
        if bool(reset):
            state = None
        pred, state = system.model(inp.cuda(), state, return_state=True)
        state = _detach_state(state)
        if s == pick:
            amp = inp[:, :, WINDOW - 1:].cpu().numpy()   # matched-input baseline
            pred_np = pred.cpu().numpy()
            wet_np = wet.numpy()
            rows = torch.nonzero(mask).squeeze(1).tolist()
            break

n_plots = min(4, len(rows))
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 3 * n_plots), sharex=True, squeeze=False)
t = np.arange(wet_np.shape[-1]) / SAMPLE_RATE
for ax, r in zip(axes[:, 0], rows[:n_plots]):
    ax.plot(t, amp[r, 0], label="Amplitude-matched (baseline in)", alpha=0.4, lw=0.5, color="gray")
    ax.plot(t, wet_np[r, 0], label="Target (wet)", alpha=0.8, lw=0.5)
    ax.plot(t, pred_np[r, 0], label="Predicted", alpha=0.8, lw=0.5)
    base_mae = float(np.mean(np.abs(amp[r, 0] - wet_np[r, 0])))
    pred_mae = float(np.mean(np.abs(pred_np[r, 0] - wet_np[r, 0])))
    pv = torch.from_numpy(pred_np[r]); tv = torch.from_numpy(wet_np[r])
    c = dm.val_dataset.cache[r]
    ax.set_title(f"{c['song']} / {c['setting']} (chunk {pick}) - "
                 f"MAE: baseline {base_mae:.4f} -> pred {pred_mae:.4f} | ESR {float(esr_metric(tv, pv)):.4f}")
    ax.set_ylabel("amp"); ax.legend(loc="lower right", fontsize=8); ax.set_ylim(-1.05, 1.05)
axes[-1, 0].set_xlabel("Time (s)")
fig.suptitle(f"Output-transformer LSTM - best val loss {callbacks[0].best_model_score:.6f}", y=1.005)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_output_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"